# SAMUDRA — slick segmenter, full GPU training

DeepLabv3+ / ResNet-50 for oil-slick segmentation in SAR imagery.
CLAUDE.md section 5.2, build order step 10.

The local machine is CPU-only, so the run that produces quotable numbers happens here.
**Runtime → Change runtime type → T4 GPU** before running anything.

## Data

| set | role | content |
|---|---|---|
| Deep-SAR SOS | primary segmentation trainer | 3,101 PALSAR + 3,354 Sentinel-1 image/mask pairs, 256×256 |
| CSIRO Sentinel-1 binary | auxiliary image-level head | 3,695 no-oil + 1,843 oil chips, 400×400 |

The binary set is **not** mixed into the segmentation loss. Its intensity statistics
differ sharply from SOS (std 8–12 against 31–49), so training segmentation across that
gap lets the network satisfy the loss by recognising which dataset an image came from
rather than where the oil is. It trains a separate classification head sharing the
backbone instead.

Neither SOS sensor encodes an oil-vs-look-alike distinction — the masks are binary.
The confusion matrix this produces is therefore **oil vs background**.

Citation for the binary set: Blondeau-Patissier, D. *CSIRO Sentinel-1 SAR image dataset
of oil- and non-oil features for machine learning (Deep Learning)*. CSIRO Data Access
Portal. https://doi.org/10.25919/4v55-dn16

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU. Training on a Colab CPU '
    'is not meaningfully faster than the local machine.')
print('torch', torch.__version__, '| cuda', torch.version.cuda)

## 1. Mount Drive

Upload the two dataset folders to Drive once, keeping the structure below. Colab's
local disk is wiped between sessions; Drive is not.

```
MyDrive/samudra/data/raw/oilspill/
  sos/train/{palsar,sentinel}/{image,label}/*.png
  sos/test/{palsar,sentinel}/{image,label}/*.png
  binary/data/{Class_0,Class_1}/*.jpg
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/samudra'
SOS   = f'{DRIVE}/data/raw/oilspill/sos'
BIN   = f'{DRIVE}/data/raw/oilspill/binary/data'
OUT   = f'{DRIVE}/artifacts'

import os
os.makedirs(OUT, exist_ok=True)
for p in (SOS, BIN):
    assert os.path.isdir(p), f'missing {p} — upload the datasets to Drive first'
print('train pairs :', len(os.listdir(f'{SOS}/train/palsar/image')) +
                       len(os.listdir(f'{SOS}/train/sentinel/image')))
print('test pairs  :', len(os.listdir(f'{SOS}/test/palsar/image')) +
                       len(os.listdir(f'{SOS}/test/sentinel/image')))

## 2. Copy the datasets to local disk

Drive I/O is slow enough to bottleneck a GPU on 6,455 small files. Copy once, then
read locally for the rest of the session.

In [ ]:
%%bash -s "$SOS" "$BIN"
set -e
mkdir -p /content/data/raw/oilspill
[ -d /content/data/raw/oilspill/sos ]    || cp -r "$1" /content/data/raw/oilspill/sos
[ -d /content/data/raw/oilspill/binary ] || { mkdir -p /content/data/raw/oilspill/binary; cp -r "$2" /content/data/raw/oilspill/binary/data; }
du -sh /content/data/raw/oilspill/*

## 3. Project code

Point `REPO` at your GitHub remote, or upload `src/` and `scripts/` to
`MyDrive/samudra/` and the fallback branch will copy them across.

In [ ]:
import os, shutil, subprocess

REPO = ''  # e.g. 'https://github.com/<you>/samudra.git'

if REPO:
    if not os.path.isdir('/content/samudra'):
        subprocess.run(['git', 'clone', REPO, '/content/samudra'], check=True)
    SRC = '/content/samudra/src'
    SCRIPTS = '/content/samudra/scripts'
else:
    os.makedirs('/content/samudra', exist_ok=True)
    for part in ('src', 'scripts'):
        s, d = f'{DRIVE}/{part}', f'/content/samudra/{part}'
        assert os.path.isdir(s), f'set REPO, or upload {part}/ to {DRIVE}/'
        if not os.path.isdir(d):
            shutil.copytree(s, d)
    SRC, SCRIPTS = '/content/samudra/src', '/content/samudra/scripts'

os.makedirs('/content/samudra/config', exist_ok=True)
print(sorted(os.listdir(SRC + '/samudra')))

## 4. Train

About 30 epochs over 6,455 pairs at batch 16 on a T4 — budget roughly 2–3 hours.
The checkpoint is written whenever oil IoU improves, so an interrupted run still
leaves a usable model.

Unlike the local machine, Colab can fetch the ImageNet weights, so the backbone starts
pretrained here. That matters: from random initialisation this dataset needs far more
epochs to reach the same IoU.

In [ ]:
!cd /content/samudra && python scripts/train_segmenter.py \
    --sos /content/data/raw/oilspill/sos \
    --binary /content/data/raw/oilspill/binary/data \
    --epochs 30 --batch-size 16 --lr 1e-4 --workers 2 \
    --checkpoint /content/seg.pt \
    --metrics /content/model_metrics.json \
    --plot /content/model_metrics.png

## 5. Save results back to Drive

Do this before the runtime disconnects — Colab reclaims idle GPU sessions and
`/content` goes with them.

In [ ]:
import json, shutil

shutil.copy('/content/seg.pt', f'{OUT}/seg.pt')
shutil.copy('/content/model_metrics.json', f'{OUT}/model_metrics.json')
shutil.copy('/content/model_metrics.png', f'{OUT}/model_metrics.png')

m = json.load(open('/content/model_metrics.json'))
seg = m['segmentation']
print('representative :', m['representative'])
print('mean IoU       : %.4f' % seg['mean_iou'])
for name, v in seg['per_class'].items():
    print('  %-11s IoU %.4f  precision %.4f  recall %.4f'
          % (name, v['iou'], v['precision'], v['recall']))
for s, v in sorted(seg.get('per_sensor', {}).items()):
    print('  [%-8s] oil IoU %.4f' % (s, v['per_class']['oil']['iou']))
print('\nSaved to', OUT)
print('Copy seg.pt to data/models/seg.pt locally — detection/segmenter.py picks it up')
print('automatically, and the pipeline falls back to the synthetic slick without it.')

In [ ]:
from IPython.display import Image, display
display(Image('/content/model_metrics.png'))